# 실전 — 실제 데이터로 검증: 실제 CVE 로 RAG 검증하기 (ch18)

ch18 의 RAG 파이프라인을 **실제 NVD CVE 데이터**로 돌린다. 진짜 CVE 설명을 임베딩해 ChromaDB 에 색인하고, 의심 로그를 질의해 **의미가 가까운 실제 CVE 를 인용(citation)** 과 함께 검색한다.

> **인터넷 필요**: 실행 시 NVD API 로 실제 CVE 를 가져오고, 임베딩 모델(all-MiniLM-L6-v2, 약 90MB)을 최초 1회 내려받는다.

핵심 관찰: (1) 임베딩 검색은 **키워드가 아니라 의미**로 매칭한다 — `UNION SELECT` 라는 표현이 CVE 설명에 없어도 'SQL injection' CVE 를 찾는다. (2) 검색 결과에는 항상 **실제 CVE ID(인용)** 가 붙어, 답변의 인용을 대조할 근거가 된다. ID가 존재해도 자산에 해당 취약점이 적용된다는 뜻은 아니다. (3) **거리 임계값**으로 '관련 CVE 없음'을 판별해, 무관한 컨텍스트를 억지로 주입하지 않는다.

## 데이터셋 이야기 — 어떻게 수집됐나 (NVD CVE)

미국 NIST 가 운영하는 **공식 취약점 데이터베이스**다. 전 세계에서 보고된 취약점이 **CVE 식별자·설명·CVSS 점수**와 함께 매일 갱신되며, 공개 API 로 누구나 조회할 수 있다(수십만 건).

**시나리오**: 의심 로그를 받았을 때 수십만 건의 실제 CVE 중 의미가 가장 가까운 것을 검색해 인용 근거로 삼을 수 있는가.

In [ ]:
# 파트별 노트북은 같은 종류의 공통 데이터·모델 작업 폴더를 사용합니다.
from pathlib import Path as _BookPath
import os as _book_os
_book_cwd = _BookPath.cwd()
if (_book_cwd.name in {"part1", "part2", "part3", "part4"}
        and _book_cwd.parent.name in {"instructor", "exercises", "solutions"}):
    _book_os.chdir(_book_cwd.parent)


In [1]:
# !pip install sentence-transformers chromadb
import urllib.request, urllib.parse, ssl, json, time
from sentence_transformers import SentenceTransformer
import chromadb

## 1. 실제 CVE 내려받기 (NVD API)

공식 NVD API 로 키워드별 실제 CVE 설명을 가져와 corpus 를 만든다. (인증 키 없이도 동작하되 rate limit 이 있어 요청 사이에 잠시 쉰다.)

In [2]:
CTX = ssl.create_default_context()
def nvd_search(keyword, n=30):
    url = ('https://services.nvd.nist.gov/rest/json/cves/2.0'
           f'?keywordSearch={urllib.parse.quote(keyword)}&resultsPerPage={n}')
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    data = json.load(urllib.request.urlopen(req, timeout=40, context=CTX))
    rows = []
    for item in data.get('vulnerabilities', []):
        cve = item['cve']
        desc = [d['value'] for d in cve['descriptions'] if d['lang'] == 'en']
        if desc:
            rows.append((cve['id'], desc[0]))
    return rows

corpus = {}
for kw in ['SQL injection', 'cross-site scripting', 'remote code execution']:
    for cid, desc in nvd_search(kw, 30):
        corpus[cid] = desc
    time.sleep(6)   # NVD rate limit 배려

ids = list(corpus)
docs = [corpus[i] for i in ids]
print(f'실제 CVE corpus: {len(ids)}건')
print('예:', ids[0], '-', docs[0][:70])

실제 CVE corpus: 89건
예: CVE-2000-1233 - SQL injection vulnerability in read.php3 and other scripts in Phorum 3


## 2. 임베딩 + ChromaDB 색인 (ch18 §18.3~18.4)

각 CVE 설명을 384차원 벡터로 변환해 벡터 DB 에 넣는다.

In [3]:
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(docs, show_progress_bar=False).tolist()

client = chromadb.Client()
col = client.get_or_create_collection('cve_real')
col.add(ids=ids, documents=docs, embeddings=embeddings)
print('색인 완료:', col.count(), '개 CVE')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

색인 완료: 89 개 CVE


## 3. 의심 로그로 검색 — 인용이 붙은 결과 (ch18 citation)

의심 로그를 질의하면, 의미가 가까운 실제 CVE 를 ID(인용)와 거리와 함께 돌려준다.

In [4]:
def retrieve(query, k=3):
    qe = model.encode([query]).tolist()
    res = col.query(query_embeddings=qe, n_results=k)
    return list(zip(res['ids'][0], res['documents'][0], res['distances'][0]))

for q in ['SQL injection in login form via UNION SELECT',
          'stored XSS in the comment field of a web board',
          'attacker uploads a webshell to run OS commands']:
    print(f'\nQ: {q}')
    for cid, doc, dist in retrieve(q):
        print(f'   [{cid}] dist={dist:.3f}  {doc[:64]}')


Q: SQL injection in login form via UNION SELECT
   [CVE-2002-0878] dist=0.849  SQL injection vulnerability in the login form for LogiSense soft
   [CVE-2001-1460] dist=1.178  SQL injection vulnerability in article.php in PostNuke 0.62 thro
   [CVE-2001-1472] dist=1.191  SQL injection vulnerability in prefs.php in phpBB 1.4.0 and 1.4.

Q: stored XSS in the comment field of a web board
   [CVE-2001-1441] dist=1.141  Cross-site scripting (XSS) vulnerability in VisualAge for Java 3
   [CVE-2001-1416] dist=1.144  Multiple cross-site scripting (XSS) vulnerabilities in the log m
   [CVE-2001-0897] dist=1.278  Cross-site scripting vulnerability in Infopop Ultimate Bulletin 

Q: attacker uploads a webshell to run OS commands
   [CVE-2006-3336] dist=1.043  TWiki 01-Dec-2000 up to 4.0.3 allows remote attackers to bypass 
   [CVE-2001-0727] dist=1.122  Internet Explorer 6.0 allows remote attackers to execute arbitra
   [CVE-2004-2560] dist=1.138  DokuWiki before 2004-10-19, when used on a web ser

## 4. 거리 임계값 — '관련 CVE 없음' 판별 (ch18 환각 방지)

검색은 항상 무언가를 돌려주지만, 거리가 너무 크면 사실상 무관하다. 임계값을 넘으면 '근거 없음'으로 처리해 LLM 에 억지 컨텍스트를 주지 않는다.

In [5]:
RESCUE_DIST = 1.2   # 이보다 멀면 관련 CVE 없음으로 판단
def rag_answer(query):
    hits = retrieve(query, k=1)
    cid, doc, dist = hits[0]
    if dist > RESCUE_DIST:
        return f'관련 CVE 없음 (최근접 거리 {dist:.2f} > {RESCUE_DIST})'
    return f'근거 CVE [{cid}] (거리 {dist:.2f}): {doc[:80]}'

print(rag_answer('SQL injection in login form'))          # 관련 CVE 있음
print(rag_answer('employee took a coffee break at noon'))  # 보안과 무관 → 근거 없음

근거 CVE [CVE-2002-0878] (거리 0.54): SQL injection vulnerability in the login form for LogiSense software including (
관련 CVE 없음 (최근접 거리 1.78 > 1.2)


## 5. 정리

1. **의미 기반 검색은 표현이 다른 자료를 찾는 데 도움이 됩니다.** 검색 결과의 관련성은 질의와 임베딩 모델에 따라 달라지므로 원문과 대조합니다.
2. **모든 결과에 실제 CVE ID 가 붙는다** — 18장의 인용 검증에 사용할 근거입니다. 이 노트북은 검색 결과를 확인하며, 후속 LLM 응답에서는 허용된 ID인지와 인용 내용이 원문에 부합하는지를 별도로 검사해야 합니다.
3. **거리 임계값이 '모름'을 만든다** — 관련 근거가 없으면 억지로 답하지 않는 것이 RAG 신뢰성의 핵심이다.

> 실제 운영에서는 NVD 전체 피드를 daily cron 으로 증분 색인하고, MITRE ATT&CK·사내 IoC 까지 같은 컬렉션에 넣어 corpus 를 키운다. 색인·검색 코드 골격은 이 노트북과 동일하다.